# Concurrent Programming Practical — Easy + Moderate
**Runtime → Change runtime type → T4 GPU before running.**

| Task | Marks |
|------|-------|
| Easy (CO2) — Column-wise Average, DS2 (1024×1024) | 20 |
| Moderate (CO4) — GPU BFS, DS8 (SNAP ca-GrQc) | 30 |

In [ ]:
!nvidia-smi
!pip install -q numba networkx

In [ ]:
import numpy as np
from numba import cuda, float32, int32
import math, time, networkx as nx
print('Imports OK')

---
## EASY (20 Marks) — CO2 | DS2
### Column-wise Average of a 1024×1024 Matrix

**Kernel design:** 1 thread per column. Thread `col` iterates all rows, accumulates sum, divides by ROWS.  
**Grid:** `ceil(1024/256) = 4` blocks × 256 threads.

In [ ]:
ROWS, COLS = 1024, 1024
np.random.seed(42)
matrix_host = np.random.rand(ROWS, COLS).astype(np.float32)

@cuda.jit
def column_avg_kernel(matrix, result, rows, cols):
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if col < cols:
        total = float32(0.0)
        for row in range(rows):
            total += matrix[row, col]
        result[col] = total / rows

matrix_device = cuda.to_device(matrix_host)
result_device = cuda.device_array(COLS, dtype=np.float32)

THREADS = 256
blocks  = math.ceil(COLS / THREADS)

# warm-up
column_avg_kernel[blocks, THREADS](matrix_device, result_device, ROWS, COLS)
cuda.synchronize()

t0 = time.perf_counter()
column_avg_kernel[blocks, THREADS](matrix_device, result_device, ROWS, COLS)
cuda.synchronize()
gpu_time = (time.perf_counter() - t0) * 1000

gpu_result = result_device.copy_to_host()
cpu_result = matrix_host.mean(axis=0)
max_error  = np.max(np.abs(gpu_result - cpu_result))

print('='*50)
print('  EASY — Column-wise Average (1024×1024)')
print('='*50)
print(f'  Grid           : {blocks} blocks × {THREADS} threads')
print(f'  GPU time       : {gpu_time:.3f} ms')
print(f'  Max abs error  : {max_error:.2e}')
print(f'  GPU avg col[0] : {gpu_result[0]:.6f}')
print(f'  CPU avg col[0] : {cpu_result[0]:.6f}')
print('  PASS' if max_error < 1e-4 else '  FAIL')

---
## MODERATE (30 Marks) — CO4 | DS8
### GPU BFS on SNAP ca-GrQc Graph

**Algorithm:** Level-synchronous BFS in CSR format.  
**Key:** `cuda.atomic.min` with `INT32_MAX` sentinel replaces `compare_and_swap` (avoids API version issues).  
**Parallelism:** 1 thread per frontier node per level.

In [ ]:
import urllib.request, gzip, os
from collections import defaultdict

URL  = 'https://snap.stanford.edu/data/ca-GrQc.txt.gz'
FILE = 'ca-GrQc.txt.gz'
if not os.path.exists(FILE):
    print('Downloading ca-GrQc ...')
    urllib.request.urlretrieve(URL, FILE)

edges = []
with gzip.open(FILE, 'rt') as f:
    for line in f:
        if line.startswith('#'): continue
        u, v = map(int, line.split())
        edges.append((u, v))
        edges.append((v, u))

nodes  = sorted(set(n for e in edges for n in e))
id_map = {n: i for i, n in enumerate(nodes)}
edges  = [(id_map[u], id_map[v]) for u, v in edges]
N      = len(nodes)

adj = defaultdict(list)
for u, v in edges:
    adj[u].append(v)

row_ptr = np.zeros(N + 1, dtype=np.int32)
for i in range(N):
    row_ptr[i + 1] = row_ptr[i] + len(adj[i])
col_idx = np.empty(row_ptr[N], dtype=np.int32)
for i in range(N):
    for j, nb in enumerate(adj[i]):
        col_idx[row_ptr[i] + j] = nb

print(f'Nodes: {N} | Directed edges: {len(edges)} | CSR ready')

In [ ]:
INT32_MAX = np.iinfo(np.int32).max

@cuda.jit
def bfs_kernel(row_ptr, col_idx, distance, frontier, next_frontier,
               frontier_size, next_size, current_level):
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid >= frontier_size[0]:
        return
    node  = frontier[tid]
    start = row_ptr[node]
    end   = row_ptr[node + 1]
    for i in range(start, end):
        nb  = col_idx[i]
        old = cuda.atomic.min(distance, nb, current_level + 1)
        if old == INT32_MAX:
            pos = cuda.atomic.add(next_size, 0, 1)
            next_frontier[pos] = nb

def gpu_bfs(source, N, row_ptr_h, col_idx_h):
    dist_h = np.full(N, INT32_MAX, dtype=np.int32)
    dist_h[source] = 0

    d_rp   = cuda.to_device(row_ptr_h)
    d_ci   = cuda.to_device(col_idx_h)
    d_dist = cuda.to_device(dist_h)
    d_front = cuda.to_device(np.array([source], dtype=np.int32))
    d_next  = cuda.device_array(N, dtype=np.int32)
    d_fs    = cuda.to_device(np.array([1], dtype=np.int32))
    d_ns    = cuda.to_device(np.array([0], dtype=np.int32))

    THREADS = 256
    level   = 0
    t0 = time.perf_counter()

    while True:
        fs = int(d_fs.copy_to_host()[0])
        if fs == 0: break
        blks = max(1, math.ceil(fs / THREADS))
        d_ns[0] = 0
        bfs_kernel[blks, THREADS](d_rp, d_ci, d_dist,
                                  d_front, d_next, d_fs, d_ns, level)
        cuda.synchronize()
        d_front, d_next = d_next, d_front
        d_fs[0] = int(d_ns.copy_to_host()[0])
        d_ns[0] = 0
        level  += 1

    elapsed = (time.perf_counter() - t0) * 1000
    dist_out = d_dist.copy_to_host()
    dist_out[dist_out == INT32_MAX] = -1
    return dist_out, elapsed, level

SOURCE = 0
dist_gpu, gpu_bfs_time, levels = gpu_bfs(SOURCE, N, row_ptr, col_idx)

G_nx = nx.Graph()
G_nx.add_edges_from([(u, v) for u, v in edges if u < v])
t0 = time.perf_counter()
bfs_lengths = nx.single_source_shortest_path_length(G_nx, SOURCE)
cpu_time = (time.perf_counter() - t0) * 1000

errors = sum(1 for node, d in bfs_lengths.items() if dist_gpu[node] != d)

print('='*50)
print('  MODERATE — GPU BFS (ca-GrQc)')
print('='*50)
print(f'  BFS depth     : {levels} levels')
print(f'  Nodes reached : {(dist_gpu >= 0).sum()}')
print(f'  GPU time      : {gpu_bfs_time:.2f} ms')
print(f'  CPU time      : {cpu_time:.2f} ms')
print(f'  Speedup       : {cpu_time/gpu_bfs_time:.2f}x')
print(f'  Mismatches    : {errors}')
print('  PASS' if errors == 0 else '  FAIL')